In [133]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [134]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch.optim as optim

In [135]:
torch.manual_seed(44)

In [136]:
df = pd.read_csv('/content/fmnist_small.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [137]:
df.shape

(6000, 785)

In [138]:
print(len(df['label'].unique()))
df['label'].unique()

10


array([9, 7, 0, 8, 1, 4, 2, 6, 5, 3])

In [139]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [140]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=44)

In [141]:
# scaling the features:
X_train = X_train/255
X_test = X_test/255

In [142]:
class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [143]:
X_train_data = CustomDataset(X_train, y_train)
X_test_data = CustomDataset(X_test, y_test)

In [144]:
X_train_dataloader = DataLoader(X_train_data, batch_size=32, shuffle=True)
X_test_dataloader = DataLoader(X_test_data, batch_size=32, shuffle=False)

In [145]:
class ToqeersNN(nn.Module):
    def __init__(self, num_features):
        super(ToqeersNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )


    def forward(self, X):
        return self.model(X)

In [146]:
epochs = 200
learning_rate = 0.1

In [147]:
model = ToqeersNN(num_features=X_train.shape[1])

In [148]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

In [149]:
for epoch in range(epochs):
    learning_rate = learning_rate * (0.95 ** epoch)
    for batch_x, batch_y in X_train_dataloader:
        optimizer.zero_grad()
        # forword pass:
        output = model(batch_x)
        # calculate loss:
        loss = loss_fn(output, batch_y)
        # claculating gradients:
        loss.backward()
        # updating weights:
        optimizer.step()
    print(f"Epoch: {epoch+1}, Loss: {loss}, learning_rate :{learning_rate}")

Epoch: 1, Loss: 0.8224020004272461, learning_rate :0.1
Epoch: 2, Loss: 0.8690072298049927, learning_rate :0.095
Epoch: 3, Loss: 0.47310614585876465, learning_rate :0.0857375
Epoch: 4, Loss: 0.5680122971534729, learning_rate :0.07350918906249998
Epoch: 5, Loss: 0.8455260992050171, learning_rate :0.059873693923837866
Epoch: 6, Loss: 0.38791051506996155, learning_rate :0.046329123015975304
Epoch: 7, Loss: 0.35688450932502747, learning_rate :0.034056162628811476
Epoch: 8, Loss: 0.5286681652069092, learning_rate :0.023782688525533214
Epoch: 9, Loss: 0.4956451952457428, learning_rate :0.015777921478822676
Epoch: 10, Loss: 0.6176905035972595, learning_rate :0.009944025698709223
Epoch: 11, Loss: 0.6651307344436646, learning_rate :0.00595385551055294
Epoch: 12, Loss: 0.23291151225566864, learning_rate :0.0033865535638032203
Epoch: 13, Loss: 0.3616807758808136, learning_rate :0.0018299583806109228
Epoch: 14, Loss: 0.6224431395530701, learning_rate :0.0009393946474176001
Epoch: 15, Loss: 0.466924

In [150]:
model.eval()

ToqeersNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [151]:
total = 0
correct = 0
with torch.no_grad():
    for batch_x, batch_y in X_test_dataloader:
        output = model(batch_x) # autometically calls forward pass from NNclass.
        _, predicted = torch.max(output.data, 1)
        total += batch_y.shape[0]
        correct += (predicted == batch_y).sum().item()

print(f"Accuracy: {correct/total}")

Accuracy: 0.8691666666666666
